# Embedding
![rag_embedding](figures/rag_embedding.png)

- 분할된 텍스트를 벡터 표현(임베딩 벡터)으로 변환한다.
- LangChain은 OpenAI, HuggingFace 등 다양한 임베딩 모델을 지원하며, 동일한 인터페이스로 사용할 수 있다.
- [임베딩모델의 메서드](https://reference.langchain.com/python/langchain/embeddings/#langchain.embeddings.init_embeddings)

    - **`embed_documents(texts: List[str])`**
        - 여러 문서를 받아 벡터화(임베딩)한다.
        - Context를 벡터화 할 때 사용한다.
    - **`embed_query(text: str)`**
        - 하나의 문자열(문서)을 받아 벡터화한다.
        - Query를 벡터화 할 때 사용한다.


In [88]:
docs = [
        "나는 고양이와 개 중 반려동물로 개를 키우고 싶습니다.",
        "이 강아지 품종은 진도개 입니다. 국제 표준으로 중대형견으로 분류되며 다리가 길어 체고가 높은 편에 속합니다.",
        "日本の市内バスの運賃は主に距離によって決まり、地域やバス会社によって異なる場合があります",                 # 일본의 시내버스 요금은 주로 거리에 따라 결정되며, 지역 및 버스 회사에 따라 다를 수 있습니다.
        "Bus fares in the United States vary from city to city, but are generally around $2.90 for a regular bus.", # 미국의 버스 요금은 도시마다 다르지만, 일반적으로 정기 버스의 경우 2.90달러 정도입니다.
        "광역버스 요금은 일반 3000원, 청소년은 1800원, 어린이 1500원 입니다.", 
]

In [89]:
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
from langchain_openai import OpenAIEmbeddings
model_name = "text-embedding-3-large"
e_model = OpenAIEmbeddings(model=model_name)
embedded_docs = e_model.embed_documents(docs)

In [ ]:
# ! uv pip install sentence_transformers

Resolved 41 packages in 3.34s
Prepared 1 package in 8.44s
Installed 4 packages in 828ms
 + joblib==1.5.3
 + scikit-learn==1.9.0
 + sentence-transformers==5.6.0
 + threadpoolctl==3.6.0


In [103]:
# huggingface의 오픈소스임베딩모델 사용 예
# https://huggingface.co/spaces/mteb/leaderboard
from langchain_huggingface import HuggingFaceEmbeddings
model_id = "codefuse-ai/F2LLM-v2-1.7B"
e_model = HuggingFaceEmbeddings(model=model_id)
embedded_docs = e_model.embed_documents(docs)

 

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [104]:
print(type(embedded_docs), len(embedded_docs))
print(type(embedded_docs[0]), len(embedded_docs[0]))

<class 'list'> 5
<class 'list'> 2048


In [105]:
embedded_docs[0][:10]

[0.003387451171875,
 -0.039306640625,
 -0.000278472900390625,
 0.0169677734375,
 0.03662109375,
 -0.0732421875,
 0.0029296875,
 0.03564453125,
 0.00982666015625,
 0.0654296875]

In [106]:
query = "개와 고양이중 무엇을 더 좋아하나요?"
# query = "요새 시내버스 요금 얼마예요?"

In [107]:
query_vector = e_model.embed_query(query)
print(type(query_vector))
query_vector[:10]

<class 'list'>


[-0.024658203125,
 -0.0269775390625,
 0.018310546875,
 0.0147705078125,
 0.0634765625,
 -0.049072265625,
 0.060302734375,
 0.0272216796875,
 0.03955078125,
 0.07568359375]

In [108]:
# 코사인 유사도 = 두 벡터간의 내적/ 각 벡터의 크기의 곱
import numpy as np
def cosin_similarity(vector1, vector2):
    """vector1과 vector2의 코사인 유사도 계산"""
    v1 = np.array(vector1)
    v2 = np.array(vector2)
    return (v1 @ v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [109]:
# v = [2, 3, 4]
# np.linalg.norm(v)

In [110]:
cosin_similarity(embedded_docs[0], query_vector)

np.float64(0.6100917848552326)

In [111]:
search_list=[]
for i, doc_vector in enumerate(embedded_docs):
    print(f"{i+1}. {cosin_similarity(doc_vector, query_vector)}")
    search_list.append((i, cosin_similarity(doc_vector, query_vector) ))

1. 0.6100917848552326
2. 0.040698408369678286
3. -0.004255048199424241
4. -0.0458986909810026
5. 0.021572702241769194


In [112]:
search_list.sort(key=lambda x: x[1], reverse=True)

In [113]:
search_list

[(0, np.float64(0.6100917848552326)),
 (1, np.float64(0.040698408369678286)),
 (4, np.float64(0.021572702241769194)),
 (2, np.float64(-0.004255048199424241)),
 (3, np.float64(-0.0458986909810026))]

In [114]:
top_k = 3
doc_idx = [idx for idx, score in search_list[:3]]
doc_idx

[0, 1, 4]

In [ ]:
# docs[0]
# docs[1]
# docs[4]

# + query = primpt -> LLM 모델에 보내 질문과 답을 받는다.

# 벡터 데이터베이스(Vector Database)

![rag_vector_store](figures/rag_vector_store.png)

- **벡터 데이터베이스는** 데이터를 고차원 벡터(임베딩)로 변환하여 저장하고, 벡터 간의 유사도를 기반으로 검색과 관리를 수행하는 특수한 형태의 데이터베이스이다.

- **주요 특징**
  - 텍스트, 이미지, 오디오 등의 비정형 데이터를 수치 벡터로 변환하여 저장
   - 코사인 유사도, 유클리드 거리 등을 이용한 벡터 간 유사도 계산을 통한 검색
   - 근사 최근접 이웃(Approximate Nearest Neighbor, ANN) 알고리즘을 통한 빠른 검색을 지원.

## 벡터 데이터베이스와 딥러닝
- 벡터 데이터베이스는 딥러닝 기술의 발전과 깊은 관련이 있다.
- 딥러닝 모델은 학습 과정에서 데이터의 특징을 추출하는 방법을 함께 학습한다. 충분한 데이터를 학습한 딥러닝 모델은 **데이터의 특성을 설명하는 특성 벡터(feature vector)를 효과적으로 생성**할 수 있다.
- 이때 추출된 특성 벡터는 고차원 데이터(RAW Data)를 저차원 공간에서 표현한 **임베딩 벡터**다.
    - > **임베딩**은 고차원 데이터를 저차원 공간으로 변환하여 표현하는 방법으로, 정보 손실을 최소화하면서 데이터 간의 의미 있는 관계를 벡터 공간에서 유지한다.
- 딥러닝 모델로 추출한 데이터의 특징(feature vector)을 임베딩 공간에 배치하면, 비슷한 데이터는 가까이, 그렇지 않은 데이터는 멀리 배치된다.
- 이러한 특성을 활용하면 임베딩 벡터 간의 거리를 계산해 유사한 데이터를 효과적으로 검색할 수 있다. 벡터 데이터베이스는 이러한 임베딩 벡터의 특성을 기반으로 개발되었다.
- 딥러닝 기술의 발전과 폭넓은 활용으로 임베딩 데이터의 사용이 증가하면서, 이를 저장하고 관리하는 기능에 특화된 데이터베이스에 대한 수요도 증가해 다양한 벡터 데이터베이스가 등장했다.

## LLM과 벡터 데이터베이스
- ChatGPT(LLM)의 등장 이후 벡터 데이터베이스는 폭발적인 주목을 받았다.
- 임베딩 벡터의 유사도를 기반으로 문서를 검색하는 RAG(Relevant Augmented Generation) 기술은 LLM의 환각(할루시네이션) 현상을 줄이고, LLM을 추가 학습하지 않고도 최신 정보를 효율적으로 활용할 수 있는 핵심 기법으로 자리 잡았다.
   


## 벡터 데이터베이스 종류
![img](figures/vector_database.png)

<<https://blog.det.life/why-you-shouldnt-invest-in-vector-databases-c0cd3f59d23c>>

### 주요 벡터 데이터베이스 종류
- **Qdrant**
    - Rust로 개발된 고성능 벡터 검색 엔진으로, 실시간 근사 최근접 이웃 검색을 제공한다.  
    - 벡터와 함께 메타데이터(payload) 필터링을 결합할 수 있어, 의미 검색과 조건 검색을 동시에 수행할 수 있다.
    - RAG·추천·시맨틱 검색 등 LLM 연계 시스템에 적합하다.
- **Pinecone**
    - 클라우드 기반의 완전 관리형 벡터 데이터베이스 서비스로, 간단한 API를 통해 벡터 데이터를 관리할 수 있다.  
    - 자동 확장성과 고가용성을 제공하며, 실시간 데이터 수집과 유사성 검색에 최적화되어 있다.
    - 가장 쉽게 시작할 수 있는 관리형 서비스를 제공한다.
- **Chroma**
    - 벡터 임베딩을 효율적으로 저장하고 검색할 수 있는 오픈소스 데이터베이스로, AI 및 머신러닝 애플리케이션에 최적화되어 있다.
    - 대규모 임베딩 저장에 최적화되어 있다.
- **FAISS**
    - Facebook AI에서 개발한 고성능 벡터 검색 라이브러리로, 고차원 벡터의 효율적인 유사성 검색을 위해 최적화되어 있다.
    - GPU를 활용해 계산 성능을 높이며, 벡터 양자화 기술을 활용하여 메모리 사용을 최적화한다.
    - 근사 최근접 이웃 검색(ANNS)에 최적화되어 있다.
- **Milvus**
    - 오픈소스 벡터 데이터베이스로, 대규모 벡터 데이터를 효율적으로 저장하고 검색할 수 있다.  
    - 분산 아키텍처를 채택하여 확장성이 뛰어나며, IVF_PQ, DiskANN 등 다양한 인덱싱 알고리즘을 지원한다.
    - 대규모 데이터셋 처리에 가장 적합한 솔루션이다.
- **Weaviate**
    - 오픈소스 벡터 데이터베이스로, 텍스트, 이미지, 오디오 등 다양한 비정형 데이터를 벡터로 저장하고 검색할 수 있다.  
    - GraphQL API를 통해 접근 가능하며, 내장된 머신러닝 모듈을 통해 가장 강력한 의미론적 검색 기능을 제공한다.
- **Elasticsearch**
    - HNSW 알고리즘을 사용하여 벡터 검색을 구현하는 검색 엔진이다.
    - 전통적인 검색 기능과 벡터 검색을 효과적으로 결합할 수 있어, 하이브리드 검색에 가장 적합하다.
- **PGVector**
    - PostgreSQL의 확장 모듈로, 벡터 데이터를 저장하고 유사성 검색을 수행할 수 있게 해준다.  
    - SQL과 통합된 벡터 연산이 가능하며, L2 거리, 코사인 거리, 내적 등 다양한 거리 측정 방식을 지원한다.


# Langchain - Vector Store 연동 
- Langchain은 다양한 벡터 데이터베이스와 연동할 수 있다.
- 벡터 데이터베이스 마다 API가 다르기 때문에, Langchain을 사용하면 동일한 interface로 사용할 수 있다.

## **VectorStore**
- Langchain이 지원하는 모든 벡터 데이터베이스는 **VectorStore** 인터페이스를 구현한다.
- 그래서 Langchain에서는 벡터 데이터베이스를 **Vector Store** 라고 한다.
- https://python.langchain.com/docs/integrations/vectorstores/

### Vector Store 연결
- Vector DB와 연결하는 메소드
- `VectorStore.from_documents()`
  - Document들을 insert 하면서 연결.
  - Database가 있으면 연결, 없으면 생성하면서 연결한다.
  - Parameter
    - documents: insert할 문서들을 list[Document]로 전달.
    - embedding model
    - vector db에 연결하기 위한 설정들을 넣어준다.
- `VectorStore()`
  - vector db와 연결만 한다.
  - Database가 있으면 연결, 없으면 생성하면서 연결한다.
  - Parameter
    - embedding model
    - vector db에 연결하기 위한 설정들을 넣어준다.
## InMemoryVectorStore
- langchain에서 제공하는 메모리 기반 벡터 데이터베이스이다.
- Data들을 Dictionary를 사용해 메모리에 저장하며, 검색 할 때 코사인 유사도(cosine similarity)를 계산하여 조회한다.

In [115]:
from langchain_core.documents import Document
d1 = Document(id=1, page_content="Apple, Pear, Watermelon", metadata={"category":"fruit"})
d2 = Document(id=2, page_content="Python, Java, C++", metadata={"category":"IT"})
d3 = Document(id=3, page_content="Footballe, Baseball, Basketball", metadata={"category":"sport"})

In [118]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embedding_model = HuggingFaceEmbeddings(model="codefuse-ai/F2LLM-v2-1.7B")
vectorstore = InMemoryVectorStore(embedding_model)
vectorstore.add_documents(documents=[d1, d2, d3])

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

['1', '2', '3']

In [121]:
query = "SQL"
result = vectorstore.similarity_search(
    query=query,
    k=2,
)

In [123]:
for doc in result:
    print(doc.page_content)

Python, Java, C++
Footballe, Baseball, Basketball


In [125]:
result = vectorstore.similarity_search_with_score(
    query=query,
    k=2,
)
for doc in result:
    print(doc)


(Document(id='2', metadata={'category': 'IT'}, page_content='Python, Java, C++'), 0.1971956296480289)
(Document(id='3', metadata={'category': 'sport'}, page_content='Footballe, Baseball, Basketball'), 0.12282100056080926)


# 실습
- "data/olympic.txt" 문서를 vector store에 저장하고 질문과 유사한 청크를 조회
  
1. text loading
2. text split
3. embedding + vector store(InMemoryVectorStore)에 저장
4. query(질의)

In [126]:
from langchain_community.document_loaders import TextLoader
path = "data/olympic.txt"

C:\Users\Playdata\AppData\Local\Temp\ipykernel_32720\2883918916.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [127]:
with open(path, "rt", encoding="utf-8") as f:
    print(f.read()[:100])

올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하


In [128]:
loader = TextLoader(path, encoding="utf-8")
docs = loader.load() # 실행시 바로 읽는다. -> list[Document]
# docs = loader.lazy_load() # 읽은 문서를 사용할 때 읽는다. -> generator[Document]
print(type(docs), len(docs))
print(type(docs[0]))
# for doc in docs:
#     print(type(doc))

<class 'list'> 1
<class 'langchain_core.documents.base.Document'>


In [129]:
doc = docs[0]
print("문서정보: doc.metadata")
print(doc.metadata)
# 필요한 정보를 추가할 수 있다. LLM에 전달할 프롬프트에 전달한 내용과 검색할 때 사용할 내용
doc.metadata["category"] = "sports"
doc.metadata["tag"] = ["올림픽", "IOC", "동계올림픽", "하계올림픽"]
print(doc.metadata)

문서정보: doc.metadata
{'source': 'data/olympic.txt'}
{'source': 'data/olympic.txt', 'category': 'sports', 'tag': ['올림픽', 'IOC', '동계올림픽', '하계올림픽']}


In [130]:
print("문서내용: doc.page_content")
print(doc.page_content[:200])

문서내용: doc.page_content
올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열


In [133]:
print("문서 식별자 -id: doc.id")
print(doc.id)


문서 식별자 -id: doc.id
None


In [134]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
path = "data/olympic.txt"
loader = TextLoader(path, encoding="UTF-8")
docs = loader.load()
print("로드한 문서개수", len(docs))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", r"[\.?!,~]", ' ', ''],
    is_separator_regex=True,
)
split_docs = splitter.split_documents(docs)
print("스플릿한 문서 개수", len(split_docs))

로드한 문서개수 1
스플릿한 문서 개수 61


In [135]:
split_docs[1].page_content

'올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다'

In [136]:
docs = loader.load_and_split(splitter)
print(len(docs))

61


In [137]:
docs[1].page_content

'올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다'

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embedding_model = HuggingFaceEmbeddings(model="codefuse-ai/F2LLM-v2-1.7B")
vectorstore = InMemoryVectorStore(embedding_model)
vectorstore.add_documents(documents=split_docs)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [ ]:
query = "SQL"
result = vectorstore.similarity_search_with_score(
    query=query,
    k=2,
)
for doc in result:
    print(doc)


## MMR(최대 한계 관련성-Maximal Marginal Relevance) 알고리즘 적용
최대 한계 관련성(Maximal Marginal Relevance, MMR) 알고리즘은 정보 검색 및 요약에서 검색 결과의 **관련성**과 **다양성**을 동시에 고려하여 최적의 결과를 제공하는 방법이다. 
이 알고리즘은 사용자 쿼리와의 관련성을 최대화하면서도 중복 정보를 최소화하여 다양한 정보를 제공하는 것을 목표로 한다.

1. **관련성과 다양성의 균형 조절**: MMR은 사용자 쿼리와 문서 간의 유사성 점수와 이미 선택된 문서들과의 다양성 점수를 조합하여 각 문서의 최종 점수를 계산한다. 이를 통해 관련성이 높으면서도 중복되지 않는 문서를 선택한다.

2. **수학적 정의**
   $$
   \text{MMR} = \lambda \cdot \text{Sim}(d, Q) - (1 - \lambda) \cdot \max_{d' \in D'} \text{Sim}(d, d')
   $$

   - $\text{Sim}(d, Q)$: 문서 $d$와 쿼리 $\text{Q}$ 사이의 유사성. (문서 유사성 계산)
   - $\max_{d' \in D'} \text{Sim}(d, d')$: 문서 $d$와 이미 선택된 문서 집합 $D'$ 중 가장 유사한 문서와의 유사성. (문서 다양성 계산)
   - $\lambda$: 유사성과 다양성의 중요도를 조절하는 매개변수(parameter)
3. **적용 분야**: MMR은 정보 검색, 추천 시스템, 문서 요약 등에서 활용된다. 특히 LLM 검색에서 성능 향상이 입증되었다.

### `vectorStore.max_marginal_relevance_search()` 메소드
  - MMR 알고리즘을 적용한 검색을 수행한다.
  - **파라미터**
    - **query**: 사용자로부터 입력받은 검색 쿼리
    - **k**: 최종적으로 선택할 문서의 수
    - **fetch\_k**: MMR 알고리즘 적용 시 고려할 상위 문서의 수
    - **lambda_mult**: 쿼리와의 유사성과 선택된 문서 간의 다양성 사이의 균형을 조절하는 매개변수. $\lambda = 1$이면 유사성만 고려하고, $\lambda = 0$이면 다양성만을 최대화한다.
    - **filter**: 검색 결과를 필터링할 조건을 지정한다.
